# 🧪 PetPlantr Testing Implementation Guide
**Turning Testing Strategy into Day-to-Day Reality**

This notebook provides practical scaffolds and code examples to implement comprehensive testing for PetPlantr's critical business paths.

## 🎯 Implementation Focus Areas
1. **Phase 1 Kickoff**: Critical business logic tests
2. **AI/ML Quality Gates**: Model regression prevention  
3. **Security & Auth**: Zero-tolerance testing
4. **Frontend Journeys**: Revenue path protection
5. **Fast Wins**: Git hooks, factories, coverage badges

## 🚀 Phase 1: Critical Foundation Setup

### Coverage Configuration & Baseline
First, let's set up pytest with coverage tracking and fail conditions.

In [ ]:
# File: pytest.ini
"""
[tool:pytest]
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*
addopts = 
    --cov=src
    --cov-fail-under=70
    --cov-report=html:htmlcov
    --cov-report=term-missing
    --cov-report=xml
    -v
    --tb=short
markers =
    critical: Critical business logic tests (must pass)
    integration: Integration tests
    e2e: End-to-end tests
    slow: Slow running tests
"""

# File: requirements-test.txt
test_requirements = """
pytest==7.4.0
pytest-cov==4.1.0
pytest-asyncio==0.21.0
pytest-mock==3.11.1
pytest-xdist==3.3.1
factory-boy==3.3.0
responses==0.23.1
freezegun==1.2.2
bandit==1.7.5
safety==2.3.5
"""

print("✅ Basic testing infrastructure configured")
print("📊 Coverage target: 70% baseline, 90% for critical components")

### Critical Business Logic Tests - STL Generation Math
The most important tests - bad geometry = failed prints = refunds

In [ ]:
# File: tests/critical/test_stl_generation.py

import pytest
import numpy as np
from unittest.mock import Mock, patch
from src.core.stl_generator import STLGenerator, MeshProcessor
from src.core.geometry_validator import GeometryValidator

class TestSTLGeneration:
    """Critical STL generation tests - MUST maintain 90%+ coverage"""
    
    def setup_method(self):
        """Setup test fixtures"""
        self.generator = STLGenerator()
        self.validator = GeometryValidator()
        
    @pytest.mark.critical
    def test_stl_volume_calculation_accuracy(self):
        """STL volume must be accurate within 1% tolerance"""
        # Test with known geometric shapes
        test_cases = [
            {'shape': 'cube', 'dimensions': [10, 10, 10], 'expected_volume': 1000},
            {'shape': 'sphere', 'radius': 5, 'expected_volume': 523.6},  # 4/3 * π * r³
            {'shape': 'cylinder', 'radius': 3, 'height': 10, 'expected_volume': 282.7}
        ]
        
        for case in test_cases:
            mesh = self.generator.create_test_mesh(case)
            calculated_volume = self.validator.calculate_volume(mesh)
            expected = case['expected_volume']
            
            tolerance = 0.01  # 1% tolerance
            assert abs(calculated_volume - expected) / expected < tolerance, \
                f"Volume calculation failed for {case['shape']}: {calculated_volume} vs {expected}"
    
    @pytest.mark.critical  
    def test_watertight_mesh_validation(self):
        """Generated meshes must be watertight for 3D printing"""
        sample_dog_image = self.load_test_image('golden_retriever_sample.jpg')
        
        stl_mesh = self.generator.generate_stl(sample_dog_image)
        
        # Critical watertight checks
        assert self.validator.is_watertight(stl_mesh), "STL mesh is not watertight"
        assert self.validator.has_no_self_intersections(stl_mesh), "Mesh has self-intersections"
        assert self.validator.volume_in_range(stl_mesh, min_vol=50, max_vol=500), "Volume out of printable range"
        
    @pytest.mark.critical
    def test_mesh_quality_metrics(self):
        """Mesh quality must meet printing standards"""
        stl_mesh = self.generator.generate_stl(self.load_test_image('test_dog.jpg'))
        
        quality_metrics = self.validator.analyze_mesh_quality(stl_mesh)
        
        # Critical quality thresholds
        assert quality_metrics['aspect_ratio'] < 10, "Poor aspect ratio"
        assert quality_metrics['edge_length_variance'] < 0.5, "Inconsistent edge lengths"
        assert quality_metrics['surface_area'] > 100, "Surface area too small for printing"
        
    def load_test_image(self, filename):
        """Helper to load test images"""
        import os
        test_data_path = os.path.join(os.path.dirname(__file__), '..', 'data', filename)
        # Return mock image data for testing
        return np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)

print("✅ STL Generation critical tests scaffolded")
print("🎯 Focus: Volume accuracy, watertight validation, quality metrics")

### Critical Pricing Logic Tests
Pricing errors = direct revenue loss

In [ ]:
# File: tests/critical/test_pricing_logic.py

import pytest
from decimal import Decimal
from unittest.mock import patch
from src.core.pricing import PricingCalculator, DiscountEngine
from src.core.models import Order, Product, Customer

class TestPricingLogic:
    """Critical pricing tests - revenue protection"""
    
    def setup_method(self):
        self.calculator = PricingCalculator()
        self.discount_engine = DiscountEngine()
        
    @pytest.mark.critical
    def test_base_pricing_calculation(self):
        """Base pricing must be consistent and accurate"""
        test_cases = [
            {'size': 'small', 'material': 'PLA', 'expected': Decimal('24.99')},
            {'size': 'medium', 'material': 'PLA', 'expected': Decimal('34.99')},
            {'size': 'large', 'material': 'PETG', 'expected': Decimal('54.99')},
        ]
        
        for case in test_cases:
            price = self.calculator.calculate_base_price(
                size=case['size'], 
                material=case['material']
            )
            assert price == case['expected'], \
                f"Price mismatch for {case['size']}/{case['material']}: {price} vs {case['expected']}"
    
    @pytest.mark.critical
    def test_discount_edge_cases(self):
        """Discount calculations must handle edge cases correctly"""
        base_price = Decimal('50.00')
        
        # Test cases that could break pricing
        edge_cases = [
            {'discount_percent': 0, 'expected': Decimal('50.00')},
            {'discount_percent': 100, 'expected': Decimal('0.00')},
            {'discount_percent': 15.5, 'expected': Decimal('42.25')},
            {'discount_percent': 0.01, 'expected': Decimal('49.995')},  # Rounding test
        ]
        
        for case in edge_cases:
            result = self.discount_engine.apply_percentage_discount(
                base_price, case['discount_percent']
            )
            assert abs(result - case['expected']) < Decimal('0.01'), \
                f"Discount calculation failed: {result} vs {case['expected']}"
    
    @pytest.mark.critical
    def test_tax_calculation_by_region(self):
        """Tax calculations must be accurate for different regions"""
        base_price = Decimal('100.00')
        
        tax_scenarios = [
            {'region': 'CA', 'expected_tax': Decimal('10.75')},  # CA sales tax
            {'region': 'NY', 'expected_tax': Decimal('8.25')},   # NY sales tax
            {'region': 'OR', 'expected_tax': Decimal('0.00')},   # No sales tax
            {'region': 'INTL', 'expected_tax': Decimal('0.00')}, # International
        ]
        
        for scenario in tax_scenarios:
            tax = self.calculator.calculate_tax(base_price, scenario['region'])
            assert tax == scenario['expected_tax'], \
                f"Tax calculation wrong for {scenario['region']}: {tax} vs {scenario['expected_tax']}"
    
    @pytest.mark.critical
    def test_bulk_order_pricing(self):
        """Large orders should receive appropriate discounts"""
        # Test volume pricing tiers
        quantities = [1, 5, 10, 25, 50]
        base_unit_price = Decimal('30.00')
        
        for qty in quantities:
            total_price = self.calculator.calculate_bulk_price(base_unit_price, qty)
            expected_discount = self.calculator.get_volume_discount_rate(qty)
            
            expected_total = base_unit_price * qty * (1 - expected_discount)
            assert abs(total_price - expected_total) < Decimal('0.01'), \
                f"Bulk pricing incorrect for qty {qty}: {total_price} vs {expected_total}"

print("✅ Pricing logic critical tests scaffolded")
print("💰 Coverage: Base pricing, discounts, taxes, bulk orders")

### Critical Payment Flow Tests
Payment bugs = fraud risk + customer trust issues

In [ ]:
# File: tests/critical/test_payment_flow.py

import pytest
import json
from unittest.mock import Mock, patch
from freezegun import freeze_time
from src.core.payment import StripePaymentProcessor, PaymentHandler
from src.core.models import Order, PaymentIntent

class TestPaymentFlow:
    """Critical payment flow tests - fraud prevention"""
    
    def setup_method(self):
        self.processor = StripePaymentProcessor()
        self.handler = PaymentHandler()
        
    @pytest.mark.critical
    def test_stripe_webhook_signature_validation(self):
        """Invalid webhook signatures must be rejected"""
        # Test with invalid signature
        invalid_payload = {
            "id": "evt_test_webhook",
            "type": "payment_intent.succeeded",
            "data": {"object": {"id": "pi_test_payment"}}
        }
        
        with pytest.raises(ValueError, match="Invalid signature"):
            self.processor.verify_webhook_signature(
                payload=json.dumps(invalid_payload),
                signature="invalid_signature",
                secret="whsec_test_secret"
            )
    
    @pytest.mark.critical
    def test_webhook_idempotency(self):
        """Duplicate webhooks should not double-charge or double-process"""
        webhook_event = {
            "id": "evt_duplicate_test",
            "type": "payment_intent.succeeded",
            "data": {"object": {"id": "pi_test_payment_123"}}
        }
        
        # Process webhook first time
        result1 = self.handler.process_webhook(webhook_event)
        assert result1['status'] == 'processed'
        
        # Process same webhook again - should be idempotent
        result2 = self.handler.process_webhook(webhook_event)
        assert result2['status'] == 'already_processed'
        
        # Verify no double-charging occurred
        order = Order.get_by_payment_intent("pi_test_payment_123")
        assert order.charge_count == 1, "Order was charged multiple times"
    
    @pytest.mark.critical
    def test_payment_timeout_handling(self):
        """Failed/timeout payments must not create completed orders"""
        with patch('src.core.payment.stripe.PaymentIntent.create') as mock_create:
            # Simulate payment timeout
            mock_create.side_effect = Exception("Payment timeout")
            
            with pytest.raises(Exception):
                payment_intent = self.processor.create_payment_intent(
                    amount=5000,  # $50.00
                    customer_id="cust_test_123"
                )
            
            # Verify no order was created in failed state
            orders = Order.get_by_customer("cust_test_123")
            incomplete_orders = [o for o in orders if o.status == 'incomplete']
            assert len(incomplete_orders) == 0, "Incomplete order created on payment failure"
    
    @pytest.mark.critical
    def test_refund_processing(self):
        """Refunds must be processed correctly and update order status"""
        # Create a successful payment first
        order = self.create_test_order_with_payment()
        
        # Process refund
        refund_result = self.processor.process_refund(
            payment_intent_id=order.payment_intent_id,
            amount=order.total_amount,
            reason="customer_request"
        )
        
        assert refund_result['status'] == 'succeeded'
        
        # Verify order status updated
        updated_order = Order.get(order.id)
        assert updated_order.status == 'refunded'
        assert updated_order.refund_amount == order.total_amount
    
    def create_test_order_with_payment(self):
        """Helper to create test order with successful payment"""
        # Mock successful payment creation
        return Order.create({
            'customer_id': 'cust_test_123',
            'total_amount': 5000,
            'status': 'completed',
            'payment_intent_id': 'pi_test_successful'
        })

print("✅ Payment flow critical tests scaffolded")
print("🔒 Coverage: Webhooks, idempotency, timeouts, refunds")

## 🤖 AI/ML Quality Gates Implementation

### Golden Dataset & Model Regression Testing
Prevents silent quality regressions when retraining or tweaking prompts.

In [ ]:
# File: tests/ai_quality/test_model_regression.py

import pytest
import numpy as np
import torch
from pathlib import Path
from src.ai.models import CLIPBreedClassifier, STLGenerator
from src.ai.metrics import calculate_psnr, calculate_chamfer_distance

class TestModelQualityGates:
    """AI/ML quality regression prevention"""
    
    @classmethod
    def setup_class(cls):
        """Load golden dataset and models once for all tests"""
        cls.golden_dataset_path = Path("tests/data/golden_dataset")
        cls.breed_classifier = CLIPBreedClassifier.load_latest()
        cls.stl_generator = STLGenerator.load_latest()
        
        # Load golden test set (small but representative)
        cls.golden_images = cls.load_golden_images()
        cls.golden_stl_metrics = cls.load_golden_stl_metrics()
    
    @pytest.mark.slow
    def test_breed_classification_accuracy_baseline(self):
        """Model accuracy must not drop below 85% on golden test set"""
        correct_predictions = 0
        total_predictions = len(self.golden_images)
        
        for image_data in self.golden_images:
            image = image_data['image']
            true_breed = image_data['breed']
            
            # Get model prediction
            predicted_breed, confidence = self.breed_classifier.predict(image)
            
            if predicted_breed == true_breed and confidence > 0.75:
                correct_predictions += 1
        
        accuracy = correct_predictions / total_predictions
        assert accuracy >= 0.85, f"Model accuracy dropped to {accuracy:.3f} (below 85% threshold)"
        
        print(f"✅ Breed classification accuracy: {accuracy:.3f}")
    
    @pytest.mark.slow
    def test_stl_generation_quality_metrics(self):
        """Generated STL quality must meet printing standards"""
        quality_failures = []
        
        for test_case in self.golden_stl_metrics:
            input_image = test_case['input_image']
            expected_metrics = test_case['metrics']
            
            # Generate STL from input
            generated_stl = self.stl_generator.generate(input_image)
            
            # Calculate quality metrics
            actual_metrics = {
                'volume': self.calculate_stl_volume(generated_stl),
                'surface_area': self.calculate_surface_area(generated_stl),
                'mesh_quality': self.assess_mesh_quality(generated_stl)
            }
            
            # Check against thresholds
            for metric, expected_value in expected_metrics.items():
                actual_value = actual_metrics[metric]
                tolerance = expected_value * 0.1  # 10% tolerance
                
                if abs(actual_value - expected_value) > tolerance:
                    quality_failures.append({
                        'case': test_case['name'],
                        'metric': metric,
                        'expected': expected_value,
                        'actual': actual_value
                    })
        
        assert len(quality_failures) == 0, f"Quality regressions detected: {quality_failures}"
    
    @pytest.mark.slow
    def test_inference_performance_benchmarks(self):
        """Model inference must complete within performance targets"""
        sample_image = self.golden_images[0]['image']
        
        # Test breed classification speed
        import time
        start_time = time.time()
        breed, confidence = self.breed_classifier.predict(sample_image)
        classification_time = time.time() - start_time
        
        assert classification_time < 5.0, f"Breed classification too slow: {classification_time:.2f}s"
        
        # Test STL generation speed
        start_time = time.time()
        stl_mesh = self.stl_generator.generate(sample_image)
        generation_time = time.time() - start_time
        
        assert generation_time < 60.0, f"STL generation too slow: {generation_time:.2f}s"
        
        print(f"✅ Performance: Classification {classification_time:.2f}s, Generation {generation_time:.2f}s")
    
    @classmethod
    def load_golden_images(cls):
        """Load golden dataset images with known breeds"""
        golden_data = [
            {
                'name': 'golden_retriever_001',
                'image': np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),  # Mock image
                'breed': 'Golden Retriever'
            },
            {
                'name': 'labrador_001', 
                'image': np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),
                'breed': 'Labrador Retriever'
            },
            # Add more golden samples...
        ]
        return golden_data
    
    @classmethod
    def load_golden_stl_metrics(cls):
        """Load expected STL quality metrics for golden dataset"""
        return [
            {
                'name': 'golden_retriever_001',
                'input_image': cls.golden_images[0]['image'],
                'metrics': {
                    'volume': 125.5,  # Expected volume in cm³
                    'surface_area': 245.8,  # Expected surface area in cm²
                    'mesh_quality': 0.95  # Expected quality score
                }
            }
            # Add more golden metrics...
        ]
    
    def calculate_stl_volume(self, stl_mesh):
        """Calculate STL volume"""
        # Implementation would calculate actual mesh volume
        return 125.0  # Mock implementation
    
    def calculate_surface_area(self, stl_mesh):
        """Calculate STL surface area"""
        return 240.0  # Mock implementation
    
    def assess_mesh_quality(self, stl_mesh):
        """Assess overall mesh quality score"""
        return 0.93  # Mock implementation

print("✅ AI/ML quality gates implemented")
print("🎯 Coverage: Accuracy baselines, quality metrics, performance benchmarks")

## 🔒 Security & Auth Testing

### Zero-Tolerance Security Testing
Catches config drifts and prevents auth failures before production.

In [ ]:
# File: tests/security/test_auth_security.py

import pytest
import jwt
from datetime import datetime, timedelta
from unittest.mock import patch, Mock
from src.auth.jwt_handler import JWTHandler
from src.auth.permissions import PermissionChecker
from src.core.models import User

class TestSecurityAuth:
    """Zero-tolerance security testing"""
    
    def setup_method(self):
        self.jwt_handler = JWTHandler()
        self.permission_checker = PermissionChecker()
        
    @pytest.mark.critical
    def test_expired_token_rejection(self):
        """Expired tokens must be rejected"""
        # Create expired token
        expired_payload = {
            'user_id': 'user_123',
            'exp': datetime.utcnow() - timedelta(hours=1)  # Expired 1 hour ago
        }
        expired_token = jwt.encode(expired_payload, 'secret', algorithm='HS256')
        
        with pytest.raises(jwt.ExpiredSignatureError):
            self.jwt_handler.decode_token(expired_token)
    
    @pytest.mark.critical
    def test_invalid_token_signature(self):
        """Tokens with invalid signatures must be rejected"""
        # Create token with wrong secret
        payload = {'user_id': 'user_123', 'exp': datetime.utcnow() + timedelta(hours=1)}
        token_wrong_secret = jwt.encode(payload, 'wrong_secret', algorithm='HS256')
        
        with pytest.raises(jwt.InvalidSignatureError):
            self.jwt_handler.decode_token(token_wrong_secret)
    
    @pytest.mark.critical
    def test_stripe_webhook_bad_signature_rejection(self):
        """Invalid Stripe webhook signatures must return 401"""
        from src.api.webhooks import stripe_webhook_handler
        from fastapi.testclient import TestClient
        from src.main import app
        
        client = TestClient(app)
        
        # Send webhook with invalid signature
        invalid_payload = {"event": "payment_intent.succeeded"}
        response = client.post(
            '/api/webhooks/stripe',
            json=invalid_payload,
            headers={'stripe-signature': 'invalid_signature'}
        )
        
        assert response.status_code == 401
        assert 'Invalid signature' in response.json()['error']
    
    @pytest.mark.critical
    def test_role_based_access_control(self):
        """Users should only access resources they have permission for"""
        # Test user without admin permissions
        regular_user = User(id='user_123', role='user')
        
        # Should NOT have admin access
        assert not self.permission_checker.has_permission(regular_user, 'admin:users:delete')
        assert not self.permission_checker.has_permission(regular_user, 'admin:orders:refund')
        
        # Should have user access
        assert self.permission_checker.has_permission(regular_user, 'user:orders:view')
        assert self.permission_checker.has_permission(regular_user, 'user:profile:edit')
        
        # Test admin user
        admin_user = User(id='admin_123', role='admin')
        assert self.permission_checker.has_permission(admin_user, 'admin:users:delete')
        assert self.permission_checker.has_permission(admin_user, 'admin:orders:refund')
    
    @pytest.mark.critical
    def test_sql_injection_prevention(self):
        """API endpoints must prevent SQL injection"""
        from src.api.orders import get_user_orders
        
        # Test with malicious input
        malicious_user_id = "1; DROP TABLE orders; --"
        
        # This should not execute SQL injection
        with pytest.raises(ValueError, match="Invalid user ID format"):
            orders = get_user_orders(malicious_user_id)
    
    @pytest.mark.critical
    def test_file_upload_security(self):
        """File uploads must validate file types and prevent malicious uploads"""
        from src.api.upload import validate_image_upload
        
        # Test malicious file types
        malicious_files = [
            {'filename': 'malware.exe', 'content_type': 'application/exe'},
            {'filename': 'script.php', 'content_type': 'application/php'},
            {'filename': 'payload.js', 'content_type': 'text/javascript'},
        ]
        
        for file_data in malicious_files:
            with pytest.raises(ValueError, match="Invalid file type"):
                validate_image_upload(file_data['filename'], file_data['content_type'])
        
        # Test valid image files
        valid_files = [
            {'filename': 'dog.jpg', 'content_type': 'image/jpeg'},
            {'filename': 'pet.png', 'content_type': 'image/png'},
        ]
        
        for file_data in valid_files:
            # Should not raise exception
            result = validate_image_upload(file_data['filename'], file_data['content_type'])
            assert result is True

print("✅ Security & Auth tests implemented")
print("🔒 Coverage: Token validation, RBAC, injection prevention, file upload security")

## 🖥️ Frontend E2E Testing

### Revenue Path Protection
Keeps the conversion funnel unbreakable during refactors.

In [ ]:
// File: cypress/e2e/critical_user_journey.cy.js

describe('Critical Revenue Path - Dog to Planter Conversion', () => {
  beforeEach(() => {
    // Setup test environment
    cy.task('db:seed')  // Seed test database
    cy.visit('/')
  })

  it('should complete full purchase journey without errors', () => {
    // 1️⃣ Upload photo
    cy.get('[data-cy=upload-zone]')
      .should('be.visible')
      .selectFile('cypress/fixtures/golden-retriever-test.jpg', {
        action: 'drag-drop'
      })

    // Wait for upload and processing
    cy.get('[data-cy=upload-progress]')
      .should('contain', '100%')
    
    // 2️⃣ Verify preview generation
    cy.get('[data-cy=breed-detection-result]')
      .should('be.visible')
      .should('contain', 'Golden Retriever')
    
    cy.get('[data-cy=3d-preview]')
      .should('be.visible')
    
    cy.get('[data-cy=preview-stl]', { timeout: 30000 })
      .should('be.visible')

    // 3️⃣ Customize options
    cy.get('[data-cy=size-selector]')
      .select('Large')
    
    cy.get('[data-cy=material-selector]')
      .select('PETG')
    
    cy.get('[data-cy=color-selector]')
      .select('Forest Green')

    // Verify price updates
    cy.get('[data-cy=total-price]')
      .should('contain', '$54.99')

    // 4️⃣ Add to cart
    cy.get('[data-cy=add-to-cart]')
      .should('not.be.disabled')
      .click()

    cy.get('[data-cy=cart-notification]')
      .should('contain', 'Added to cart')

    // 5️⃣ Proceed to checkout
    cy.get('[data-cy=view-cart]').click()
    
    cy.get('[data-cy=cart-item]')
      .should('contain', 'Golden Retriever Planter')
    
    cy.get('[data-cy=checkout-button]').click()

    // 6️⃣ Fill shipping information
    cy.get('[data-cy=shipping-form]').within(() => {
      cy.get('[name=firstName]').type('John')
      cy.get('[name=lastName]').type('Doe')
      cy.get('[name=email]').type('john.doe@test.com')
      cy.get('[name=address]').type('123 Test Street')
      cy.get('[name=city]').type('Test City')
      cy.get('[name=state]').select('CA')
      cy.get('[name=zipCode]').type('90210')
    })

    // 7️⃣ Payment with Stripe (sandbox)
    cy.get('[data-cy=payment-section]').within(() => {
      // Use Stripe test card
      cy.get('[data-cy=stripe-card-number]')
        .type('4242424242424242')
      
      cy.get('[data-cy=stripe-expiry]')
        .type('12/28')
      
      cy.get('[data-cy=stripe-cvc]')
        .type('123')
      
      cy.get('[data-cy=stripe-zip]')
        .type('90210')
    })

    // 8️⃣ Submit payment
    cy.get('[data-cy=submit-payment]')
      .should('not.be.disabled')
      .click()

    // 9️⃣ Wait for payment processing
    cy.get('[data-cy=payment-processing]')
      .should('be.visible')

    // 🎯 Verify successful completion
    cy.get('[data-cy=success-message]', { timeout: 30000 })
      .should('be.visible')
      .should('contain', 'Order confirmed')

    cy.get('[data-cy=order-number]')
      .should('be.visible')
      .should('match', /^ORD-\d+$/)

    cy.get('[data-cy=download-stl-link]')
      .should('be.visible')
      .should('have.attr', 'href')

    // Verify email confirmation would be sent
    cy.get('[data-cy=email-confirmation]')
      .should('contain', 'john.doe@test.com')
  })

  it('should handle payment failures gracefully', () => {
    // Quick path to checkout
    cy.get('[data-cy=upload-zone]')
      .selectFile('cypress/fixtures/test-dog.jpg')
    
    cy.get('[data-cy=add-to-cart]', { timeout: 30000 }).click()
    cy.get('[data-cy=checkout-button]').click()

    // Fill minimal info
    cy.get('[name=email]').type('test@test.com')
    
    // Use declined test card
    cy.get('[data-cy=stripe-card-number]')
      .type('4000000000000002')  // Stripe test card that will be declined
    
    cy.get('[data-cy=stripe-expiry]').type('12/28')
    cy.get('[data-cy=stripe-cvc]').type('123')

    cy.get('[data-cy=submit-payment]').click()

    // Should show error message
    cy.get('[data-cy=payment-error]')
      .should('be.visible')
      .should('contain', 'payment was declined')

    // Should allow retry
    cy.get('[data-cy=retry-payment]')
      .should('be.visible')
  })

  it('should work on mobile devices', () => {
    // Test mobile viewport
    cy.viewport('iphone-x')

    // Mobile-specific upload flow
    cy.get('[data-cy=mobile-upload-button]').click()
    cy.get('[data-cy=camera-capture]').should('be.visible')
    
    // Continue with file upload instead of camera
    cy.get('[data-cy=file-upload-option]').click()
    cy.get('[data-cy=mobile-file-input]')
      .selectFile('cypress/fixtures/mobile-dog-photo.jpg')

    // Verify mobile UI elements
    cy.get('[data-cy=mobile-preview]').should('be.visible')
    cy.get('[data-cy=mobile-size-selector]').should('be.visible')
    
    // Complete mobile checkout
    cy.get('[data-cy=mobile-add-to-cart]').click()
    cy.get('[data-cy=mobile-checkout]').click()

    // Mobile payment form should be responsive
    cy.get('[data-cy=mobile-payment-form]')
      .should('be.visible')
      .should('have.css', 'display', 'block')
  })
})

console.log("✅ Frontend E2E critical journey implemented")
console.log("🎯 Coverage: Upload→Preview→Customize→Payment→Success")

## ⚡ Fast Wins Implementation

### Git Hooks Setup
Stops bad commits before they hit CI.

In [ ]:
#!/bin/bash
# File: setup_git_hooks.sh

echo "🔧 Setting up Git hooks for PetPlantr..."

# Install husky for git hooks
npm install --save-dev husky
npx husky install

# Pre-commit hook - run critical tests
npx husky add .husky/pre-commit "npm run test:critical"

# Pre-push hook - run full test suite
npx husky add .husky/pre-push "npm run test:all"

# Create package.json scripts
echo "📝 Adding test scripts to package.json..."

cat << 'EOF' >> package.json.append
{
  "scripts": {
    "test:critical": "pytest tests/critical/ -v --tb=short",
    "test:unit": "pytest tests/unit/ -v",
    "test:integration": "pytest tests/integration/ -v",
    "test:security": "bandit -r src/ && safety check",
    "test:frontend": "npm run test:unit:frontend && npm run test:e2e:critical",
    "test:all": "npm run test:critical && npm run test:unit && npm run test:security",
    "test:coverage": "pytest --cov=src --cov-report=html --cov-report=term",
    "test:ai": "pytest tests/ai_quality/ -v -m slow"
  }
}
EOF

echo "✅ Git hooks configured!"
echo "🚀 Now commits will be blocked if critical tests fail"

### Test Data Factory Setup
Consistent in-memory models - no flaky DB state.

In [ ]:
# File: tests/factories.py

import factory
from factory.fuzzy import FuzzyChoice, FuzzyInteger, FuzzyDecimal
from datetime import datetime, timedelta
from src.core.models import User, Order, Product, PaymentIntent

class UserFactory(factory.Factory):
    """Factory for creating test users"""
    class Meta:
        model = User
    
    id = factory.Sequence(lambda n: f"user_{n}")
    email = factory.Sequence(lambda n: f"user{n}@test.com")
    first_name = factory.Faker('first_name')
    last_name = factory.Faker('last_name')
    role = FuzzyChoice(['user', 'admin'])
    created_at = factory.LazyFunction(datetime.utcnow)
    is_active = True

class ProductFactory(factory.Factory):
    """Factory for creating test products"""
    class Meta:
        model = Product
    
    id = factory.Sequence(lambda n: f"prod_{n}")
    name = factory.Faker('word')
    size = FuzzyChoice(['small', 'medium', 'large'])
    material = FuzzyChoice(['PLA', 'PETG', 'ABS'])
    base_price = FuzzyDecimal(10.00, 100.00, 2)
    is_active = True

class OrderFactory(factory.Factory):
    """Factory for creating test orders"""
    class Meta:
        model = Order
    
    id = factory.Sequence(lambda n: f"ord_{n}")
    user = factory.SubFactory(UserFactory)
    product = factory.SubFactory(ProductFactory)
    status = FuzzyChoice(['pending', 'processing', 'completed', 'cancelled'])
    total_amount = FuzzyDecimal(20.00, 200.00, 2)
    created_at = factory.LazyFunction(datetime.utcnow)

class PaymentIntentFactory(factory.Factory):
    """Factory for creating test payment intents"""
    class Meta:
        model = PaymentIntent
    
    id = factory.Sequence(lambda n: f"pi_test_{n}")
    order = factory.SubFactory(OrderFactory)
    amount = factory.LazyAttribute(lambda obj: obj.order.total_amount)
    status = FuzzyChoice(['requires_payment_method', 'succeeded', 'canceled'])
    created_at = factory.LazyFunction(datetime.utcnow)

# Usage examples in tests:
def test_example_using_factories():
    """Example of using factories in tests"""
    # Create test user
    user = UserFactory()
    assert user.email.endswith('@test.com')
    
    # Create order with specific attributes
    order = OrderFactory(status='completed', total_amount=50.00)
    assert order.status == 'completed'
    assert order.total_amount == 50.00
    
    # Create related objects
    completed_order = OrderFactory(
        user=UserFactory(role='admin'),
        product=ProductFactory(material='PETG'),
        status='completed'
    )
    assert completed_order.user.role == 'admin'
    assert completed_order.product.material == 'PETG'

print("✅ Test data factories implemented")
print("🏭 Usage: UserFactory(), OrderFactory(), ProductFactory()")

### Coverage Badge Setup
Visual progress tracking for the team.

In [ ]:
# File: .github/workflows/coverage.yml

name: Coverage Report
on: [push, pull_request]

jobs:
  coverage:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.9'
          
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install -r requirements-test.txt
          
      - name: Run tests with coverage
        run: |
          pytest --cov=src --cov-report=xml --cov-report=term
          
      - name: Upload coverage to Codecov
        uses: codecov/codecov-action@v3
        with:
          token: ${{ secrets.CODECOV_TOKEN }}
          file: ./coverage.xml
          flags: unittests
          name: codecov-umbrella
          
      - name: Update coverage badge
        if: github.ref == 'refs/heads/main'
        run: |
          # Generate coverage badge
          pip install coverage-badge
          coverage-badge -o coverage.svg
          
      - name: Commit coverage badge
        if: github.ref == 'refs/heads/main'
        uses: stefanzweifel/git-auto-commit-action@v4
        with:
          commit_message: 'Update coverage badge'
          file_pattern: coverage.svg

## 📊 Implementation Checklist & Next Steps

### Phase 1: Critical Foundation (Week 1-2)
Ready-to-use scaffolds provided above for immediate implementation.

In [ ]:
# Implementation Checklist
implementation_checklist = {
    "Phase 1 - Critical Foundation": {
        "pytest_config": "✅ pytest.ini with coverage settings",
        "stl_math_tests": "✅ Volume, watertight, quality tests",
        "pricing_tests": "✅ Base pricing, discounts, tax calculations", 
        "payment_tests": "✅ Stripe webhooks, idempotency, refunds",
        "git_hooks": "✅ Pre-commit hooks with critical tests",
        "test_factories": "✅ User, Order, Product factories",
        "priority": "🚨 IMPLEMENT FIRST - Revenue protection"
    },
    
    "Phase 2 - AI/ML Quality Gates": {
        "golden_dataset": "✅ Small representative test set",
        "model_regression": "✅ Accuracy baselines, quality metrics",
        "performance_benchmarks": "✅ Inference speed requirements",
        "weekly_quality_check": "🔄 Automated model validation",
        "priority": "🤖 PREVENT MODEL REGRESSIONS"
    },
    
    "Phase 3 - Security & Auth": {
        "auth_tests": "✅ Token validation, RBAC",
        "security_scanning": "✅ Bandit, safety checks",
        "input_validation": "✅ SQL injection, file upload tests",
        "stripe_security": "✅ Webhook signature validation",
        "priority": "🔒 ZERO-TOLERANCE SECURITY"
    },
    
    "Phase 4 - Frontend E2E": {
        "critical_journey": "✅ Upload→checkout→success flow",
        "payment_failure": "✅ Graceful error handling",
        "mobile_testing": "✅ Responsive design validation",
        "cypress_setup": "✅ Test configuration and fixtures",
        "priority": "🖥️ PROTECT CONVERSION FUNNEL"
    },
    
    "Fast Wins": {
        "git_hooks": "✅ husky pre-commit setup",
        "coverage_badge": "✅ Codecov integration",
        "test_scripts": "✅ npm test commands",
        "ci_pipeline": "🔄 GitHub Actions workflow",
        "priority": "⚡ IMMEDIATE PRODUCTIVITY GAINS"
    }
}

# Next Steps Prioritization
next_steps = [
    "1. 🚨 Implement critical business logic tests (STL, pricing, payments)",
    "2. ⚡ Set up git hooks to prevent bad commits",
    "3. 🤖 Create golden dataset for AI quality gates", 
    "4. 🔒 Add security tests for auth and webhooks",
    "5. 🖥️ Build critical E2E journey test",
    "6. 📊 Set up coverage tracking and badges",
    "7. 🔄 Integrate all tests into CI pipeline"
]

print("📋 IMPLEMENTATION ROADMAP")
print("=" * 50)
for phase, items in implementation_checklist.items():
    print(f"\n{phase}:")
    for key, value in items.items():
        if key != "priority":
            print(f"  {value}")
        else:
            print(f"  🎯 {value}")

print("\n🎯 IMMEDIATE NEXT STEPS:")
print("=" * 30)
for step in next_steps:
    print(step)

print("\n✅ ALL SCAFFOLDS PROVIDED - READY TO IMPLEMENT!")

## 🎯 Success Metrics & Monitoring

Track these metrics weekly to ensure testing strategy is effective:

In [ ]:
# Weekly Testing Metrics Dashboard
testing_metrics = {
    "Critical Coverage": {
        "target": "≥90%",
        "components": ["STL math", "pricing", "payments", "auth"],
        "measurement": "pytest --cov=src/critical --cov-report=term"
    },
    
    "CI Pipeline Health": {
        "target": "≥95% pass rate",
        "components": ["critical tests", "security scans", "E2E smoke"],
        "measurement": "GitHub Actions success rate"
    },
    
    "Model Quality Gates": {
        "target": "≥85% accuracy",
        "components": ["breed classification", "STL quality", "performance"],
        "measurement": "Weekly golden dataset validation"
    },
    
    "Security Posture": {
        "target": "0 critical vulnerabilities",
        "components": ["bandit scans", "dependency checks", "auth tests"],
        "measurement": "Security scan results"
    },
    
    "Development Velocity": {
        "target": "<5min test runtime",
        "components": ["critical tests", "unit tests", "integration"],
        "measurement": "Test execution time"
    }
}

# Print metrics dashboard
print("📊 TESTING SUCCESS METRICS")
print("=" * 40)
for category, details in testing_metrics.items():
    print(f"\n{category}:")
    print(f"  🎯 Target: {details['target']}")
    print(f"  📦 Components: {', '.join(details['components'])}")
    print(f"  📏 Measurement: {details['measurement']}")

print("\n🎉 TESTING STRATEGY COMPLETE!")
print("Ready to protect every money-, security-, and brand-critical path! 🛡️")